# Graphora Quickstart - Transform Documents into Knowledge Graphs in 5 Minutes

**Graphora** is an AI-powered knowledge graph extraction tool that transforms unstructured documents into structured, queryable knowledge graphs. With automatic schema detection and in-memory processing, you can go from document to graph visualization in minutes without any infrastructure setup.

**What you'll learn:**
- Extract entities and relationships from documents using AI
- Visualize knowledge graphs directly in Colab
- Export to Neo4j for advanced graph analytics

**Links:**
- [Try the Visual Schema Builder](https://demo.graphora.io)
- [GitHub Repository](https://github.com/graphora/graphora-api)
- [Documentation](https://github.com/graphora/graphora-api/blob/main/README.md)

In [ ]:
# Install Graphora CLI with all dependencies
!pip install graphora[cli] -q
print("✓ Graphora CLI installed successfully!")

## Step 1: Upload Your Document

Upload any text document (PDF, DOCX, TXT, MD, etc.) to extract knowledge from. For this demo, you can use:
- A research paper
- A technical specification
- Meeting notes
- Any document with structured information

In [ ]:
from google.colab import files

# Upload document
print("Please upload your document...")
uploaded = files.upload()

# Get the uploaded filename
doc_path = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {doc_path}")
print(f"  Size: {len(uploaded[doc_path]) / 1024:.2f} KB")

## Step 2: Extract Knowledge Graph

Graphora uses AI to automatically:
- Detect entities (people, organizations, concepts, etc.)
- Identify relationships between entities
- Infer an optimal schema for your document
- Structure everything into a graph

**No configuration needed** - the auto-schema detection will analyze your document and create an appropriate ontology.

In [ ]:
import json

# Extract knowledge graph with auto-schema detection
print("Extracting knowledge graph...\n")
!graphora extract "{doc_path}" --output graph.json

print("\n✓ Knowledge graph extracted successfully!")
print("  Output saved to: graph.json")

## Step 3: Explore the Results

Let's examine what Graphora extracted from your document. We'll look at:
- How many nodes (entities) were found
- What types of entities were detected
- How many relationships connect them

In [ ]:
from collections import Counter

# Load the extracted graph
with open('graph.json', 'r') as f:
    graph = json.load(f)

# Calculate statistics
nodes = graph.get('nodes', [])
edges = graph.get('edges', [])

node_types = Counter([node.get('type', 'Unknown') for node in nodes])
edge_types = Counter([edge.get('type', 'Unknown') for edge in edges])

print("📊 Knowledge Graph Summary")
print("=" * 50)
print(f"\nTotal Nodes: {len(nodes)}")
print(f"Total Relationships: {len(edges)}")

print("\n📦 Node Types:")
for node_type, count in node_types.most_common():
    print(f"  • {node_type}: {count}")

print("\n🔗 Relationship Types:")
for edge_type, count in edge_types.most_common():
    print(f"  • {edge_type}: {count}")

# Show sample nodes
print("\n📝 Sample Entities:")
for i, node in enumerate(nodes[:5]):
    name = node.get('properties', {}).get('name', node.get('id', 'Unknown'))
    node_type = node.get('type', 'Unknown')
    print(f"  {i+1}. [{node_type}] {name}")

## Step 4: Visualize the Graph

Now let's create an interactive visualization of your knowledge graph. Nodes are colored by their type, and edges show the relationships between entities.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Create directed graph
G = nx.DiGraph()

# Add nodes with types
node_colors = {}
color_map = plt.cm.Set3(np.linspace(0, 1, len(node_types)))
type_to_color = {node_type: color_map[i] for i, node_type in enumerate(node_types.keys())}

for node in nodes:
    node_id = node.get('id')
    node_type = node.get('type', 'Unknown')
    name = node.get('properties', {}).get('name', node_id)
    G.add_node(node_id, label=name, node_type=node_type)
    node_colors[node_id] = type_to_color.get(node_type, 'gray')

# Add edges
edge_labels = {}
for edge in edges:
    source = edge.get('source')
    target = edge.get('target')
    rel_type = edge.get('type', '')
    if source and target:
        G.add_edge(source, target)
        edge_labels[(source, target)] = rel_type

# Create visualization
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw nodes
node_color_list = [node_colors.get(node, 'gray') for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_color_list, 
                       node_size=1500, alpha=0.9, edgecolors='black', linewidths=2)

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, 
                       arrowsize=20, arrowstyle='->', width=2, alpha=0.6,
                       connectionstyle='arc3,rad=0.1')

# Draw labels
node_labels = {node: G.nodes[node].get('label', node)[:20] for node in G.nodes()}
nx.draw_networkx_labels(G, pos, node_labels, font_size=8, font_weight='bold')

# Draw edge labels
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, font_color='red')

# Create legend
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=type_to_color[node_type], 
                              markersize=10, label=node_type)
                   for node_type in node_types.keys()]
plt.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.title(f"Knowledge Graph: {doc_path}\n{len(nodes)} entities, {len(edges)} relationships", 
          fontsize=16, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\n✓ Visualized {len(nodes)} nodes and {len(edges)} relationships")

## Step 5: Export for Neo4j (Optional)

Want to do advanced graph analytics? Export your knowledge graph to Neo4j format:
- Download the graph as JSON for programmatic access
- Download Cypher import statements for Neo4j
- Run complex graph queries and algorithms
- Build graph-powered applications

In [ ]:
# Export as Cypher statements for Neo4j import
print("Exporting to Neo4j Cypher format...\n")
!graphora export graph.json --format cypher --output import.cypher

print("\n✓ Export complete!\n")
print("Downloading files...")

# Download both files
files.download('graph.json')
files.download('import.cypher')

print("\n✓ Files downloaded successfully!")
print("\nTo import into Neo4j:")
print("  1. Open Neo4j Browser")
print("  2. Copy contents of import.cypher")
print("  3. Paste and execute in Neo4j")

## Next Steps

You've successfully extracted and visualized a knowledge graph from your document! Here's what you can do next:

### 1. Customize Your Schema
Use the visual schema builder to define custom ontologies:
- [Visual Schema Builder](https://demo.graphora.io) - Design schemas with a drag-and-drop interface
- Define specific entity types and relationships for your domain
- Export schemas as YAML and use with `graphora extract --schema schema.yaml`

### 2. Infer Schema from Documents
```bash
!graphora schema infer document.pdf --output schema.yaml
!graphora extract document.pdf --schema schema.yaml --output graph.json
```

### 3. Advanced Features
- **Batch Processing**: Extract from multiple documents
- **Custom Entity Resolution**: Merge duplicate entities intelligently
- **Graph Enrichment**: Add external knowledge to your graph
- **API Integration**: Use Graphora's REST API for production workflows

### 4. Resources
- [API Documentation](https://github.com/graphora/graphora-api/blob/main/docs/api.md)
- [Self-Hosting Guide](https://github.com/graphora/graphora-api/blob/main/docs/deployment.md)
- [GitHub Discussions](https://github.com/graphora/graphora-api/discussions) - Ask questions and share use cases
- [Example Notebooks](https://github.com/graphora/graphora-api/tree/main/examples) - More advanced tutorials

### 5. Deploy to Production
Ready to scale? Deploy Graphora with:
- Docker containers for microservices
- Neo4j for graph database backend
- Kubernetes for orchestration
- Cloud providers (AWS, GCP, Azure)

---

**Questions or feedback?** Open an issue on [GitHub](https://github.com/graphora/graphora-api/issues) or join the discussion!